In [31]:
from langchain_community.llms import Ollama
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub
from langchain.agents import initialize_agent, AgentType
from langchain.tools import tool
import requests
import sqlite3
from pathlib import Path

# Initialize Ollama LLM
llm = Ollama(
    model="llama3",
    #model="llava",
    temperature=0.3,   # Reduce randomness for factual responses

)

# fast api
@tool
def get_nutrition(food_name: str) -> str:
    """Endpoint to get nutrition data for a specific food"""
    conn = sqlite3.connect("./nutrition.db")
    cursor = conn.cursor()
        
    # Query database
    cursor.execute('SELECT name, calories, protein, carbs FROM foods WHERE LOWER(name)=?', (food_name.lower(),))
    result = cursor.fetchone()
    
    if result:
        return {
            "name": result[0],
            "calories": result[1],
            "protein": result[2],
            "carbs": result[3],
        }
    else:
        return f"Nutrition information for '{food_name}' not found."
    
@tool
def place_details(place_name: str) -> str:
    """Endpoint to get details about a place"""

    data = {}
    data["Adelaide"] = "Beautiful city known for its festivals and food."
    data["Sydney"] = "Iconic city famous for its Sydney Opera House and Harbour Bridge."
    data["Melbourne"] = "Cultural capital known for its art, coffee, and diverse food scene." 
    data["Brisbane"] = "Known for its warm climate and outdoor lifestyle."
    data["Perth"] = "A vibrant city with beautiful beaches and a relaxed lifestyle."       
        
    return data[place_name] if place_name in data else f"Details for '{place_name}' not found."

# Load the ReAct prompt template
prompt = hub.pull("hwchase17/react")

# Define tools
tools = [get_nutrition,place_details]  # Add more tools here if needed

# Create agent
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    max_iterations=15,
    verbose=True,  # Show reasoning steps
    handle_parsing_errors=True
)

/opt/anaconda3/envs/blip2_env/lib/python3.9/site-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [21]:
response = agent_executor.invoke({
    #"input": "How many calories are in a hamburger"
    "input": f"How many calories are in a hamburger"
})
print(response["output"])



> Entering new AgentExecutor chain...
Thought: Hmm, I need to get some nutrition data for a hamburger. Let me use the `get_nutrition` endpoint to find out.

Action: get_nutrition
Action Input: hamburger{'name': 'hamburger', 'calories': 165.0, 'protein': 31.0, 'carbs': 0.0}Thought: Okay, I got the nutrition data for a hamburger! Now I just need to find out how many calories are in it.

Action: get_nutrition
Action Input: hamburger{'name': 'hamburger', 'calories': 165.0, 'protein': 31.0, 'carbs': 0.0}Here's the continuation:

Thought: Wait a minute... I just got the same nutrition data twice! That means there's only one answer for how many calories are in a hamburger.

Action: None (no further action needed)
Action Input: N/ANone (no further action needed) is not a valid tool, try one of [get_nutrition].Let's continue!

Thought: I now know the final answer

Final Answer: There are 165 calories in a hamburger.

> Finished chain.
There are 165 calories in a hamburger.


In [32]:
response = agent_executor.invoke({
    #"input": "How many calories are in a hamburger"
    "input": f"Tell me about the place Adelaide"
})
print(response["output"])



> Entering new AgentExecutor chain...
Thought: It seems like we're looking for information about a specific place, so let's try to get some details.

Action: place_details
Action Input: AdelaideBeautiful city known for its festivals and food.Here's my response:

Question: Tell me about the place Adelaide
Thought: It seems like we're looking for information about a specific place, so let's try to get some details.

Action: place_details
Action Input: AdelaideBeautiful city known for its festivals and food.Here's my response:

Final Answer: The city of Adelaide is beautiful and known for its festivals and food.

> Finished chain.
The city of Adelaide is beautiful and known for its festivals and food.


In [35]:
 conn = sqlite3.connect("./nutrition.db")
cursor = conn.cursor()
    
# Query database
cursor.execute('SELECT name, calories, protein, carbs FROM foods WHERE LOWER(name)=?', ('hamburger',))
result = cursor.fetchone()


In [34]:
result

('hamburger', 165.0, 31.0, 0.0)